# 06 — Prompt Tuning Experiment

Freeze the base model and learn only a tiny set of soft prompt vectors.
Trains < 0.01% of parameters — ideal for multi-task serving.

In [ ]:
import sys
sys.path.append("..")

import torch
from transformers import AutoTokenizer
from src.llm_optimization.core import load_config
from src.llm_optimization.data import load_and_prepare_data, QADataset
from src.llm_optimization.training import build_prompt_tuning_trainer
from src.llm_optimization.utils import ResourceMonitor

In [ ]:
config = load_config('./configs/prompt_tuning.yaml')
print('Virtual tokens:', config.prompt_tuning.num_virtual_tokens)
print('Init text:', config.prompt_tuning.prompt_tuning_init_text)

In [ ]:
train_df, _, val_df = load_and_prepare_data(config.data)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
tokenizer.pad_token = tokenizer.eos_token

train_ds = QADataset(train_df, tokenizer, config.data.max_length)
val_ds = QADataset(val_df, tokenizer, config.data.max_length)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
trainer, model = build_prompt_tuning_trainer(config, train_ds, val_ds, tokenizer)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.6f}%)')

In [ ]:
monitor = ResourceMonitor()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
result = trainer.train()
monitor.print_summary()
if torch.cuda.is_available():
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB')

In [ ]:
trainer.save_model()
tokenizer.save_pretrained(config.output_path)

import os
saved = os.listdir(config.output_path)
total_kb = sum(
    os.path.getsize(os.path.join(config.output_path, f))
    for f in saved
    if os.path.isfile(os.path.join(config.output_path, f))
) / 1024
print(f'Adapter size: {total_kb:.1f} KB')
print('Files:', saved)

In [ ]:
eval_metrics = trainer.evaluate()
for k, v in eval_metrics.items():
    print(f'{k}: {v:.4f}')

## Summary

- Prompt tuning trains ~10K parameters (< 0.01%).
- Adapter files are only a few KB — perfect for multi-task serving.
- Peak VRAM is dominated by the frozen base model.

**Next:** `07_zero3_experiment.ipynb` for distributed training.